# Training Setup
Works on **RunPod** and **local Windows/Linux**. Run cells top-to-bottom. Setup cells (1–8) only need to run once.

## 0. Detect platform

In [ ]:
import platform, sys, os

IS_WINDOWS = platform.system() == "Windows"
IS_RUNPOD  = os.path.exists("/workspace")

if IS_RUNPOD:
    PROJECT = "/workspace/Pattern_Recog_Project"
else:
    PROJECT = os.path.abspath(os.path.join(os.getcwd(), ".."))

print(f"Platform : {platform.system()}")
print(f"RunPod   : {IS_RUNPOD}")
print(f"Project  : {PROJECT}")

## 1. Pull latest code (RunPod only)

In [ ]:
import subprocess
if IS_RUNPOD:
    subprocess.run(["git", "-C", PROJECT, "pull"], check=True)
else:
    print("Local machine — skipping git pull. Make sure your code is up to date.")

## 2. Java for METEOR/CIDEr metrics
- **RunPod/Linux**: installs automatically
- **Windows**: install manually from https://adoptium.net (Temurin JRE 17)

In [ ]:
import shutil
if IS_WINDOWS:
    if shutil.which("java"):
        print("Java found:", shutil.which("java"))
    else:
        print("Java not found. Install from https://adoptium.net then restart the notebook.")
else:
    os.system("apt-get update -q && apt-get install -y --no-install-recommends default-jre-headless -q")
    os.system("java -version")

## 3. PyTorch
- **RunPod** (CUDA 11.8): pins `torch==2.1.0+cu118`
- **Local RTX 4070** (CUDA 12.x): installs latest torch with cu124

In [ ]:
if IS_RUNPOD:
    # Pin to CUDA 11.8 — prevents accidental upgrades that break CUDA on RunPod
    %pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 \
        --index-url https://download.pytorch.org/whl/cu118 -q
else:
    # Local — install latest torch with CUDA 12.4 support (RTX 4070)
    %pip install torch torchvision \
        --index-url https://download.pytorch.org/whl/cu124 -q

## 4. Other Python dependencies

In [ ]:
%pip install timm pycocoevalcap pycocotools triton -q

## 5. mamba-ssm CUDA kernel (VMamba speedup)
- **Linux only** — compiles from source, takes **20-40 min**
- **Windows**: not supported — VMamba will use Triton fallback (still works)
- Skip this cell if you don't want to wait

In [ ]:
if IS_WINDOWS:
    print("mamba-ssm requires Linux + CUDA. Skipping on Windows.")
    print("VMamba will use Triton/PyTorch fallback — training still works.")
else:
    %pip install mamba-ssm --no-build-isolation

## 6. Cache pretrained weights

In [ ]:
import os
if IS_RUNPOD:
    os.environ["HF_HOME"] = "/workspace/hf_cache"
    os.makedirs("/workspace/hf_cache", exist_ok=True)
else:
    cache = os.path.join(PROJECT, ".hf_cache")
    os.environ["HF_HOME"] = cache
    os.makedirs(cache, exist_ok=True)
print("HF_HOME:", os.environ["HF_HOME"])

## 7. Working directory + path

In [ ]:
import sys, os
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
print("Working directory:", os.getcwd())

## 8. Download dataset

In [ ]:
%run src/data/make_data.py
%run src/data/preprocess_annotations.py

## 9. Verify setup

In [ ]:
import torch, shutil
from src.models.encoder_vmamba import WITH_MAMBA_SSM

print(f"Platform      : {platform.system()}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CUDA kernel   : {WITH_MAMBA_SSM}  <- True = fast VMamba selective scan")
print(f"Java (metrics): {shutil.which('java') is not None}  <- True = METEOR/CIDEr will work")
print(f"Flickr8k data : {os.path.exists('data/flickr8k_annotations.json')}  <- True = dataset ready")

## 10. Config — edit before training

In [ ]:
ENCODER    = "vmamba_small"   # vit_base | vit_small | vmamba_small
DECODER    = "transformer"    # transformer | mamba | mamba3
DATASET    = "flickr8k"       # flickr8k | mscoco
BATCH_SIZE = 16 if not IS_WINDOWS else 32   # 4070 has more VRAM than RunPod cu118 setup
EPOCHS     = 10
LR         = 4e-4
WORKERS    = 4 if IS_WINDOWS else 8
FREEZE_ENC = True

## 11. Train

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "src/models/train.py",
    "--dataset",    DATASET,
    "--encoder",    ENCODER,
    "--decoder",    DECODER,
    "--batch_size", str(BATCH_SIZE),
    "--lr",         str(LR),
    "--epochs",     str(EPOCHS),
    "--workers",    str(WORKERS),
]
if FREEZE_ENC:
    cmd.append("--freeze_encoder")

print("Running:", " ".join(cmd))
subprocess.run(cmd)